# Capítulo 12

Complete as células marcadas com `TODO`, substituindo cada `____` pelo código pedido. Execute as células em ordem.

In [2]:
import pandas as pd
import numpy as np
from numpy.linalg import svd
import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from scipy.cluster import hierarchy

%matplotlib inline

## 12.5.1 Análise de componentes principais

Vamos usar os dados `USArrests`, que contêm os 50 estados americanos nas linhas, e, nas colunas, tipos de crimes (cada entrada corresponde ao número de crimes por 100.000 habitantes), além da população urbana de cada estado.

In [ ]:
USArrests = pd.read_csv('datasets/USArrests.csv', index_col=0)

Faça uma exploraçao basica no conjunto de dados. Em primeiro lugar,como varia a proporção de cada crime (como fração da população)?

In [ ]:
# TODO:

Porque é uma boa idea usar `StandardScaler()`? O que faz o comando?

In [ ]:
# TODO: padronize USArrests com StandardScaler (média zero e variância um em cada coluna).
X = pd.DataFrame(____, index=USArrests.index, columns=USArrests.columns)
X.shape

Aqui, $n=50$ e $p=4$. Os componentes principais são dados por $Z = \tilde{X}W$, onde $W \in \mathbb{R}^{p \times p}$ são os loadings.

In [ ]:
# TODO: ajuste PCA em X e obtenha os loadings com o atributo components_.
# Em components_, cada linha é um componente; aqui, cada coluna deve ser um componente.
pca_loadings = pd.DataFrame(____, index=USArrests.columns, columns=['V1', 'V2', 'V3', 'V4'])
pca_loadings

E podemos checar que os valores acima são exatamente os autovetores de $X^T X$:

In [ ]:
# TODO: calcule os autovalores e autovetores de X^T X com np.linalg.eig.
eigenvalues, eigenvectors = ____
eigenvectors

In [ ]:
eigenvalues

Porque o sinal pode mudar na respostas emcima?

Agora, vamos obter os quatro primeiros componentes principais de `X` (ou seja, $z_i = \tilde{X}w_i$ para $i=1, 2, 3, 4$).

In [ ]:
# TODO: ajuste pca em X e calcule os scores com um único método.
pca = PCA()
df_plot = pd.DataFrame(____, columns=['PC1', 'PC2', 'PC3', 'PC4'], index=X.index)
df_plot

E novamente podemos checar que esses valores são apenas a projeção de $X$ na base de autovetores:

In [ ]:
# TODO: projete X na base de autovetores (Z = XW) e mostre as últimas linhas.
(____).tail()

Apesar de termos recuperado uma matriz $Z=X W \in \mathbb{R}^{50 \times 4}$ com o mesmos tamanho da matriz original, as colunas agora têm um significado próprio, qual? 

In [ ]:
fig , ax1 = plt.subplots(figsize=(9,7))

ax1.set_xlim(-3.5,3.5)
ax1.set_ylim(-3.5,3.5)

# TODO: escreva o nome de cada estado na posição (PC1, -PC2).
# O sinal de PC2 é invertido apenas para igualar à figura do livro.
for i in df_plot.index:
    ax1.annotate(i, (____, ____), ha='center')

# Plot reference lines
ax1.hlines(0,-3.5,3.5, linestyles='dotted', colors='grey')
ax1.vlines(0,-3.5,3.5, linestyles='dotted', colors='grey')

ax1.set_xlabel('First Principal Component')
ax1.set_ylabel('Second Principal Component')

# Plot Principal Component loading vectors, using a second y-axis.
ax2 = ax1.twinx().twiny()

ax2.set_ylim(-1,1)
ax2.set_xlim(-1,1)
ax2.tick_params(axis='y', colors='orange')
ax2.set_xlabel('Principal Component loading vectors', color='orange')

# Plot labels for vectors. Variable 'a' is a small offset parameter to separate arrow tip and text.
a = 1.07
for i in pca_loadings[['V1', 'V2']].index:
    ax2.annotate(i, (pca_loadings.V1.loc[i]*a, -pca_loadings.V2.loc[i]*a), color='orange')

# TODO: desenhe uma seta da origem até (V1, -V2) para cada variável.
for i in pca_loadings.index:
    ax2.arrow(0, 0, ____, ____)
plt.show()

O que captura o primeiro componente principal? e o que o segundo componente principal ?

Para decidir quantos PCs são suficientes para capturar os dados apropriadamente, podemos olhar para a variância explicada por um certo número de PCs. Isso é uma alternativa ao que fizemos com PCR, que foi usar a performance preditiva de um modelo baseado em PCs para decidir.

In [ ]:
# TODO: use os atributos do objeto pca ajustado.
print(f"Variância explicada por cada PC: {____}")
print(f"Proporção de variância explicada por cada PC: {____}")

Como vimos, a variância explicada é apenas uma normalização dos autovalores:

In [ ]:
# TODO: normalize os autovalores para que somem 1.
____

Para ajudar a decidir quantos PCs usar, podemos simplesmente plotar os valores acima e procurar algum momento me que a variância explicada adicional é negligente.

In [ ]:
plt.figure(figsize=(7,5))

# TODO: plote a proporção de variância explicada por PC e a proporção acumulada (np.cumsum).
plt.plot([1,2,3,4], ____, '-o', label='Individual component')
plt.plot([1,2,3,4], ____, '-s', label='Cumulative')

plt.ylabel('Proportion of Variance Explained')
plt.xlabel('Principal Component')
plt.xlim(0.75,4.25)
plt.ylim(0,1.05)
plt.xticks([1,2,3,4])
plt.legend(loc=2)
plt.show()

Por exemplo, se queremos garantir ao menos 80% da variância sendo explicada pelos PCs, precisamos incluir 2 PCs.